# **Artificial Intelligence - Lab 04**

***Practice Tasks:***

*Task 1:*

In [1]:
class DLSAgent:
    def __init__(self, goal, depth_limit):
        self.goal = goal
        self.depth_limit = depth_limit

    def formulate_goal(self, percept):
        if percept == self.goal:
            return "Goal reached"
        return "Searching"

    def dls(self, graph, start):
        stack = [(start, 0)]
        visited = set()
        while stack:
            node, depth = stack.pop()
            print(f"Visiting: {node}, Depth: {depth}")
            if node == self.goal:
                return f"Goal {self.goal} found within depth limit"
            if depth < self.depth_limit:
                visited.add(node)
                for neighbor in reversed(graph.get(node, [])):
                    if neighbor not in visited:
                        stack.append((neighbor, depth + 1))
        return "Goal not found within depth limit"

    def act(self, percept, graph):
        goal_status = self.formulate_goal(percept)
        if goal_status == "Goal reached":
            return f"Goal {self.goal} found"
        else:
            return self.dls(graph, percept)

In [2]:
import heapq

class UCSAgent:
    def __init__(self, goal):
        self.goal = goal

    def utility(self, cost):
        return -cost

    def ucs(self, graph, start):
        pq = [(0, start, [])]
        visited = set()

        while pq:
            cost, node, path = heapq.heappop(pq)

            if node in visited:
                continue

            visited.add(node)
            path = path + [node]

            print(f"Visiting: {node}, Cost so far: {cost}")

            if node == self.goal:
                return f"Goal {self.goal} found with minimum cost {cost}. Path: {path}"

            for neighbor, edge_cost in graph.get(node, []):
                if neighbor not in visited:
                    heapq.heappush(pq, (cost + edge_cost, neighbor, path))

        return "Goal not found"

    def act(self, percept, graph):
        return self.ucs(graph, percept)

In [3]:
class Environment:
    def __init__(self, graph):
        self.graph = graph

    def get_percept(self, node):
        return node

def run_agent(agent, environment, start_node):
    percept = environment.get_percept(start_node)
    action = agent.act(percept, environment.graph)
    print(action)

In [4]:
graph = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F'],
    'D': [],
    'E': [],
    'F': []
}

environment = Environment(graph)
agent = DLSAgent(goal='E', depth_limit=2)
run_agent(agent, environment, 'A')

Visiting: A, Depth: 0
Visiting: B, Depth: 1
Visiting: D, Depth: 2
Visiting: E, Depth: 2
Goal E found within depth limit


In [5]:
graph2 = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 1)],
    'D': [],
    'E': [],
    'F': []
}

environment = Environment(graph2)
agent = UCSAgent(goal='F')
run_agent(agent, environment, 'A')


Visiting: A, Cost so far: 0
Visiting: B, Cost so far: 1
Visiting: D, Cost so far: 3
Visiting: C, Cost so far: 4
Visiting: F, Cost so far: 5
Goal F found with minimum cost 5. Path: ['A', 'C', 'F']


*Task 2:*

In [6]:
import heapq

def ucs(distance_matrix):
    num_cities = len(distance_matrix)
    pq = [(0, 0, frozenset([0]))]
    visited = {}
    while pq:
        cost, current_city, visited_set = heapq.heappop(pq)

        if len(visited_set) == num_cities and current_city == 0:
            return cost
        if (current_city, visited_set) in visited and visited[(current_city, visited_set)] <= cost:
            continue
        visited[(current_city, visited_set)] = cost

        for next_city in range(num_cities):
            if next_city not in visited_set:
                new_visited_set = visited_set | frozenset([next_city])
                new_cost = cost + distance_matrix[current_city][next_city]
                heapq.heappush(pq, (new_cost, next_city, new_visited_set))

    return None

distance_matrix = [
    [0, 10, 15, 20],
    [10, 0, 25, 35],
    [15, 25, 0, 30],
    [20, 35, 30, 0]
]

shortest_cost = ucs(distance_matrix)
print(f"Shortest possible route cost: {shortest_cost}")

Shortest possible route cost: None


*Task 3:*

In [7]:
tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': ['H'],
    'E': [],
    'F': ['I'],
    'G': [],
    'H': [],
    'I': []
}

def dlsTree(node, goal, depth, path):
    if depth == 0:
        return False
    if node == goal:
        path.append(node)
        return True
    if node not in tree:
        return False
    for child in tree[node]:
        if dlsTree(child, goal, depth - 1, path):
            path.append(node)
            return True
    return False

def idsTree(start, goal, max_depth):
    for depth in range(max_depth + 1):
        print(f"Depth: {depth}")
        path = []
        if dlsTree(start, goal, depth, path):
            print("Path to goal:", " > ".join(reversed(path)))
            return
    print("Goal not found")

start_node = 'A'
goal_node = 'I'
max_search_depth = 5
idsTree(start_node, goal_node, max_search_depth)

Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4
Path to goal: A > C > F > I


In [8]:
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F', 'G'],
    'D': ['B', 'H'],
    'E': ['B'],
    'F': ['C', 'I'],
    'G': ['C'],
    'H': ['D'],
    'I': ['F']
}

def dlsGraph(node, goal, depth, path, visited):
    if depth == 0:
        return False
    if node == goal:
        path.append(node)
        return True
    if node in visited:
        return False
    visited.add(node)
    if node not in graph:
        return False
    for child in graph[node]:
        if dlsGraph(child, goal, depth-1, path, visited):
            path.append(node)
            return True
    visited.remove(node)
    return False

def idsGraph(start, goal, max_depth):
    for depth in range(max_depth + 1):
        print(f"Depth: {depth}")
        path = []
        visited = set()
        if dlsGraph(start, goal, depth, path, visited):
            print("Path to goal:", " > ".join(reversed(path)))
            return
    print("Goal not found")

start_node = 'A'
goal_node = 'I'
max_search_depth = 5
idsGraph(start_node, goal_node, max_search_depth)

Depth: 0
Depth: 1
Depth: 2
Depth: 3
Depth: 4
Path to goal: A > C > F > I


***Lab Tasks:***

*Task 1:*

In [9]:
graph = {
    'Tehran': ['Baghdad', 'Istanbul'],
    'Baghdad': ['Cairo'],
    'Istanbul': ['Berlin'],
    'Cairo': ['Washington'],
    'Berlin': ['Washington'],
    'Washington': []
}

def bfs(graph, start, goal):
    visited = []
    queue = []
    
    visited.append(start)
    queue.append([start])   # store path instead of single node

    while queue:
        path = queue.pop(0)
        node = path[-1]

        if node == goal:
            print("\nGoal found")
            print("Drone path to peace:", path)
            break

        for neighbour in graph[node]:
            if neighbour not in visited:
                visited.append(neighbour)
                new_path = path + [neighbour]
                queue.append(new_path)

start_node = 'Tehran'
goal_node = 'Washington'

bfs(graph, start_node, goal_node)


Goal found
Drone path to peace: ['Tehran', 'Baghdad', 'Cairo', 'Washington']


*Task 2:*

In [ ]:
graph2 = {
    'Earth': {
        'Moon_Base': 10,
        'Orbital_Platform': 5
    },
    'Orbital_Platform': {
        'Moon_Base': 2,
        'Mars': 60
    },
    'Moon_Base': {
        'Mars': 50
    },
    'Mars': {}
}

def ucs(graph, start, goal):
    frontier = [(start, 0)]
    visited = set()
    cost_so_far = {start: 0}
    came_from = {start: None}

    while frontier:
        frontier.sort(key=lambda x: x[1])
        current_node, current_cost = frontier.pop(0)

        if current_node in visited:
            continue

        visited.add(current_node)

        if current_node == goal:
            path = []
            while current_node is not None:
                path.append(current_node)
                current_node = came_from[current_node]
            path.reverse()
            print("Goal found")
            print("Path:", path)
            print("Total Cost:", current_cost, "units")
            return

        for neighbor, cost in graph[current_node].items():
            new_cost = current_cost + cost
            if neighbor not in cost_so_far or new_cost < cost_so_far[neighbor]:
                cost_so_far[neighbor] = new_cost
                came_from[neighbor] = current_node
                frontier.append((neighbor, new_cost))
    print("Goal not found")

start_node = 'Earth'
goal_node = 'Mars'
ucs(graph2, start_node, goal_node)

Goal found
Path: ['Earth', 'Orbital_Platform', 'Moon_Base', 'Mars']
Total Cost: 57 units


*Task 3:*

In [ ]:
graph = {
    'Entrance': ['Hallway_A', 'Hallway_B'],
    'Hallway_A': ['Storage'],
    'Hallway_B': ['Target'],
    'Storage': ['Deep_Vault'],
    'Target': [],
    'Deep_Vault': []
}

def dls(node, goal, limit):
    print("Visiting:", node, "Limit:", limit)

    if node == goal:
        print("\nMission Accomplished!")
        return True

    if limit == 0:
        return False

    for neighbor in graph[node]:
        found = dls(neighbor, goal, limit - 1)
        if found:
            return True
    return False

start_node = 'Entrance'
goal_node = 'Target'
limit = 2

dls(start_node, goal_node, limit)

Visiting: Entrance Limit: 2
Visiting: Hallway_A Limit: 1
Visiting: Storage Limit: 0
Visiting: Hallway_B Limit: 1
Visiting: Target Limit: 0

Mission Accomplished!


True

*Task 4:*

In [14]:
graph = {
    'Main_Box': ['Zone_A', 'Zone_B'],
    'Zone_A': ['Switch_1', 'Switch_2'],
    'Zone_B': ['Breaker_101'],
    'Switch_1': [],
    'Switch_2': [],
    'Breaker_101': []
}

def dls_2(node, goal, limit):
    if node == goal:
        return True

    if limit == 0:
        return False

    for neighbor in graph[node]:
        if dls_2(neighbor, goal, limit - 1):
            return True

    return False

def ids(start, goal, max_depth):
    for depth in range(max_depth + 1):
        print("Checking Depth", depth, "...")
        found = dls_2(start, goal, depth)
        if found:
            print("Breaker found; Lights restored")
            return

    print("Breaker not found")

ids('Main_Box', 'Breaker_101', 2)

Checking Depth 0 ...
Checking Depth 1 ...
Checking Depth 2 ...
Breaker found; Lights restored
